Verify _just_ the loop structure.
That's kind of interesting.
It's _all__ inductive theorem proving

using spacer?
ric3?




I feel like maybe I've paid my firmware tax again. Maybe I need some smaller, easier more pleasurable stuff to keep moving.
lamport examples
interrupts
s2nbignum


verilog?
frama-c
os-dev
knuckledragger2
ray tracer
simd


micropatching?
Make a model of original assembly, get it from decompilation



Guide how to use

The t2 = 2 the whole time example. 
Lifting byte model to word model.
little interpreter, calculator

leroy compiler. Stack machines
nand2tetris

time shifting

ibex to sail verification

Try the s2n-bignum examples again
A relational example?

Verus printing



verilog printing

k-induction
wf_ult - termination


Implies(ForAll([t], ULT(f(t+1), f(t))), Exists(N, f(N) == 0))

cond = is_in_loop
ForAll([t], cond(t), ULE(f(t+1), f(t))), Exists(N, Not(cond(t)))



Not(ForAll([t], ULT(f[t+1], f[t])))

Exists(t, UGE(f[t+1], f[t]))

Exists(t, t < 2**32)

Exists(x, UGE(f[x], f[x+1])) # a weird pigeonhole principle.

It's kind of a mix of a piegeonhole and well foundedness

ForAll(x, Exists(y, y != x, UGE(f[x], f[y])))

https://github.com/verilog-proof/VerilLean



In [4]:
from kdrag.all import *
hr,hr1 = Ints("hr hr1")
def Inv(init, trans, inv, sorts=None): # it's own judgement?
    if sorts is None:
        sorts = [init.domain(i) for i in range(init.arity())]
    vs = [Const(f"v_{i}", s) for i, s in enumerate(sorts)]
    vs1 = [Const(f"vnext_{i}", s) for i, s in enumerate(sorts)]
    return And(ForAll(vs, init(*vs), inv(*vs)), ForAll(vs + vs1, inv(*vs), trans(*vs, *vs1), inv(*vs1)))
hcini = define("hcini", [hr], And(hr >= 1, hr <= 12))
hctrans = define("hctrans", [hr, hr1], hr1 == If(hr == 12, 1, hr + 1))

prove(Inv(hcini, hctrans, hcini), unfold=1)

|= And(ForAll(v_0, Implies(hcini(v_0), hcini(v_0))),
    ForAll([v_0, vnext_0],
           Implies(And(hcini(v_0), hctrans(v_0, vnext_0)),
                   hcini(vnext_0))))

In [ ]:
def Inv(vs, vs1, init=None, trans=None, inv=None, stutter=False): # it's own judgement?
    assert init is not None and trans is not None and inv is not None
    if not kd.utils.free_in(vs1, inv):
        raise Exception("invariant contains next step variables")
    if not kd.utils.free_in(vs1, init):
        raise Exception("initial state contains next step variables")
    if kd.utils.free_in(vs, inv):
        raise Exception("inv does not contain vs. Proving a trivial invariant?")
    if kd.utils.free_in(vs, inv):
        raise Exception("inv does not contain vs. Proving a trivial invariant?")
    if isinstance(vs, smt.ExprRef):
        vs = [vs]
    if isinstance(vs1, smt.ExprRef):
        vs1 = [vs1]
    if stutter:
        trans = Or(trans, And(*[v == v1 for v, v1 in zip(vs, vs1)]))
    # support both lambda and variable forms
    if not isinstance(init, smt.ExprRef):
        init = init(*vs)
    if not isinstance(trans, smt.ExprRef): # also support step functions?
        if trans.range() == BoolSort():
            trans = trans(*vs, *vs1)
        elif trans.arity() == 1 and len(vs) == 1 and len(vs1) == 1:
            trans = smt.Eq(vs1[0], trans(vs[0]))
        else:
            raise Exception("trans must be a relation or a step function of one variable")
    if not isinstance(inv, smt.ExprRef):
        inv = inv(*vs)
    return And(ForAll(vs, init, inv), ForAll(vs + vs1, inv, trans, smt.substitute(inv, *zip(vs, vs1))))

prove(Inv(hr, hr1, 
       init=And(hr >= 1, hr <= 12),
       trans=hr1 == If(hr == 12, 1, hr + 1),
       inv=hr >= 1))

def model_check(vs, vs1, init, trans, inv, N=10):
    # actually do the solve or just return the formula.
    # x(t) ? x(0) == vs. But maybe bringing in a theory we don't want.
    tvs = [[FreshConst(v.sort(), prefix=v.decl().name() + str(t)) for v in vs] for t in range(N)]
    vnow = vs
    s.add(init)
    s = Solver()
    #for vnext in tvs:
    #    s

def spacer(vs, vs1, init, trans, inv):
    # try to strehgnthen the invariant to make it inductive.
    inv2 = 

def Refine(vs, P, Q):
    """
    P <= Q
    """
    return ForAll(vs, Q, P)


In [2]:
from kdrag.all import *
hr,hr1 = Ints("hr hr1")
def Inv(init, trans, inv, sorts=None): # it's own judgement?
    if sorts is None:
        sorts = [init.domain(i) for i in range(init.arity())]
    vs = [Const(f"v_{i}", s) for i, s in enumerate(sorts)]
    vs1 = [Const(f"vnext_{i}", s) for i, s in enumerate(sorts)]
    return And(ForAll(vs, init(*vs), inv(*vs)), ForAll(vs + vs1, inv(*vs), trans(*vs, *vs1), inv(*vs1)))

Inv(
    Lambda(hr, And(hr >= 1, hr <= 12)), 
    Lambda([hr, hr1], hr1 == If(hr == 12, 1, hr + 1)),
    Lambda(hr, hr >= 1)
    )


AttributeError: 'QuantifierRef' object has no attribute 'arity'

# riscv formal

In [115]:
%%file /tmp/add1.s
# riscv add1
.text
.globl add1
add1:
    addi a0, a0, 1
done:
    ret


Writing /tmp/add1.s


In [124]:
! riscv64-linux-gnu-gcc -o /tmp/add1.o /tmp/add1.s -c

In [126]:
! riscv64-linux-gnu-objdump -d /tmp/add1.o


/tmp/add1.o:     file format elf64-littleriscv


Disassembly of section .text:

0000000000000000 <add1>:
   0:	0505                	addi	a0,a0,1

0000000000000002 <done>:
   2:	8082                	ret


Hmm. The pcode model of this will expose reading and writing from giants rams....



In [100]:
from kdrag.all import *
import kdrag.contrib.yosys as yosys
path = "/home/philip/Documents/fpga/riscv-formal/"
import kdrag.printers.verilog as vlg


x,y,z,a = BitVecs("x y z a", 32)
m = Array("m", BitVecSort(32), BitVecSort(32))

w  = BitVec("w", 64)
with open("/tmp/mymod.v", "w") as f:
    f.write(vlg.to_verilog("mymod", [x,y], {z : x + y, 
                                              w : Concat(x,y),
                                             #a : m[x] + m[y] Yeah, memory doesn't work
                                             }))
    


In [ ]:
%%file /tmp/mymod_tb.v

`timescale 1ns / 1ps
module mymod_tb;

    // Inputs
    reg [31:0] x;
    reg [31:0] y;

    // Outputs
    wire [31:0] z;
    wire [63:0] w;

    // Instantiate the Unit Under Test (UUT)
    mymod uut (
        .x(x), 
        .y(y), 
        .z(z), 
        .w(w)
    );

    initial begin        
    end
endmodule


Overwriting /tmp/mymod_tb.v


In [114]:
! ebmc /tmp/mymod.v -p "z == x + y"

Converting
Type-checking Verilog::mymod
Synthesis Verilog::mymod
No engine given, attempting heuristic engine selection
Attempting tautology check
Attempting transition property
Using MiniSAT 2.2.1 with simplifier
Checking transition properties
Checking command-line assertion using transition property engine
SAT checker: instance is UNSATISFIABLE
UNSAT: property holds for all transitions

** Results:
[command-line assertion] always mymod.z == mymod...: PROVED (transition property)


In [110]:
! ebmc /tmp/mymod.v /tmp/mymod_tb.v  --top mymod_tb --trace

Converting
Type-checking Verilog::mymod_tb
Synthesis Verilog::mymod_tb
No engine given, attempting heuristic engine selection
Attempting tautology check
Using MiniSAT 2.2.1 with simplifier
SAT checker: instance is SATISFIABLE
Attempting transition property
Attempting completeness threshold
Generating Decision Problem
Using MiniSAT 2.2.1 with simplifier
Properties
Solving with propositional reduction
SAT checker: instance is SATISFIABLE
SAT: path found
SAT checker inconsistent: instance is UNSATISFIABLE
UNSAT: No path found within bound

** Results:
[mymod_tb.0.assert.1] mymod_tb.z == 3: REFUTED
Counterexample:

Transition system state 0
----------------------------------------------------
  mymod_tb.x = 5 (00000000000000000000000000000101)
  mymod_tb.y = 6 (00000000000000000000000000000110)
  mymod_tb.z = 11 (00000000000000000000000000001011)
  mymod_tb.w = 64'h500000006
  mymod_tb.uut.x = 5 (00000000000000000000000000000101)
  mymod_tb.uut.y = 6 (00000000000000000000000000000110)
  my

In [ ]:

x = BitVec("x", 32)
P = Array("P", BitVecSort(32), BoolSort())
@PTheorem(ForAll([x, P]), P(0), )
def wf_ult()



In [ ]:
f = Array("f", IntSort(), IntSort())
t = Int("t")
@PTheorem(ForAll([f, t], ForAll(t, t >= 0, f[t+1] < f[t]), Implies(t > 0, f[t] <= f[0] - t)))
def decreasing(l):
    f,t = l.fixes()
    l.intros()
    l.induct(t)
    l.auto()
    l.auto()
    n = l.fix("t")
    l.intros()
    l.specialize(0,n)
    l.split(-1)
    l.intros()
    l.have(f[n+1] < f[n], by=[])
    l.have(f[n] < f[0] - n)



    


Next Goal:
 [f!15095, t!15096, t!15099];
[Implies(t!15099 >= 0,
        f!15095[t!15099 + 1] < f!15095[t!15099]),
 t!15099 >= 0,
 Implies(t!15099 > 0,
        f!15095[t!15099] <= f!15095[0] - t!15099),
 t!15099 + 1 > 0,
 f!15095[t!15099 + 1] < f!15095[t!15099]]
?|= f!15095[t!15099] < f!15095[0] - t!15099


In [48]:
from kdrag.all import *
P = Array('P', IntSort(), BoolSort())
t = Int('t')
@Theorem(ForAll([P, t], t >= 0, 
                And(P(0), P(1)), ForAll([t], t >= 2, And(P(t-2), P(t-1)), And(P(t-1), P(t))), 
                And(P(t), P(t+1))))
def induct2(l):
    P,t = l.fixes()
    l.induct(t)
    l.auto()
    l.auto()
    l.auto()

@Theorem(ForAll([P, t], t >= 0, 
                P(0), P(1), ForAll([t], t >= 0, P(t), P(t + 1), P(t + 2)), 
                And(P(t), P(t+1))))
def induct2_pair(l):
    P,t = l.fixes()
    l.induct(t)
    l.auto()
    l.auto()
    l.auto()

@Theorem(ForAll([P, t], t >= 0, 
                P(0), P(1), ForAll([t], t >= 0, P(t), P(t + 1), P(t + 2)), 
                P(t)))
def induct2(l):
    P,t = l.fixes()
    l.auto(by=induct2_pair(P, t))



"""
@PTheorem(ForAll([P, t], t >= 0, 
                P(0), P(1), ForAll([t], t >= 0, P(t), P(t+1), P(t+2)), 
                P(t)))
def induct2(l):
    P,t = l.fixes()
    l.induct(t)
    l.auto()
    l.auto()
    #l.auto()
"""


In [6]:
import kdrag.printers.rust as rust

help(rust.compile_rust)

Help on function compile_rust in module kdrag.printers.rust:

compile_rust(fun_name, fun_code, dir='/tmp/kdrag_rust')



In [ ]:
from kdrag.all import *
def to_verus(e : smt.ExprRef):
    


In [1]:
from kdrag.all import *

PC = kd.Enum("PC", ["mycpy", "Loop", "done"])
BV32 = smt.BitVecSort(32)
State = kd.Struct("State", ("pc", PC), ("ram", smt.ArraySort(BV32, smt.BitVecSort(8))), ("len", BV32), ("src", BV32), ("dst", BV32))

@kd.reflect.reflect
def step(st : State) -> State:
    if st.pc == PC.mycpy:
        if st.len == smt.BitVecVal(0, 32):
            return st._replace(pc = PC.done)
        else:
            return st._replace(pc = PC.Loop)
    elif st.pc == PC.Loop:
        st = st._replace(ram=smt.Store(st.ram, st.dst, st.ram[st.src]))
        st = st._replace(len=st.len-1)
        st = st._replace(src=st.src+1)
        st = st._replace(dst=st.dst+1)
        if st.len == 0:
            return st._replace(pc = PC.done)
        else:
            return st._replace(pc = PC.Loop)
    elif st.pc == PC.done:
        return st
    else:
        return st #kd.Undef(State) # unreachable
    
step
import kdrag.printers.rust as rust
mod = rust.VerusModule("mystep")
mod.datatypes.append(State)
mod.datatypes.append(PC)
#mod.datatypes.append(PC)
mod.add_defn(step)
print(mod.to_str())
mod.check("/tmp/mystep.rs")

mod mystep {
use vstd::prelude::*;
verus! {
struct State { pc : PC, ram : Map<u32, u8>, len : u32, src : u32, dst : u32 }
enum PC { mycpy {  },Loop {  },done {  } }
spec fn step(st_KDBANG_120: State) -> State { (if (st_KDBANG_120.pc == (PC::mycpy{})) { (if (0x0 == st_KDBANG_120.len) { (State { pc : (PC::done{}) , ram : st_KDBANG_120.ram , len : st_KDBANG_120.len , src : st_KDBANG_120.src , dst : st_KDBANG_120.dst  }) } else { (State { pc : (PC::Loop{}) , ram : st_KDBANG_120.ram , len : st_KDBANG_120.len , src : st_KDBANG_120.src , dst : st_KDBANG_120.dst  }) }) } else { (if (st_KDBANG_120.pc == (PC::Loop{})) { (if ((State { pc : st_KDBANG_120.pc , ram : st_KDBANG_120.ram.insert(st_KDBANG_120.dst, (st_KDBANG_120.ram[st_KDBANG_120.src])) , len : ((State { pc : st_KDBANG_120.pc , ram : st_KDBANG_120.ram.insert(st_KDBANG_120.dst, (st_KDBANG_120.ram[st_KDBANG_120.src])) , len : st_KDBANG_120.len , src : st_KDBANG_120.src , dst : st_KDBANG_120.dst  }).len.wrapping_sub(0x1)) , src : ((State {

enum fields do not work.
Could do a inline pattern match with default or with panic in other branch. Ugly.
Or define accessors.
impl Foo {
    yada yada yada.
}
It's too bad

Recollect up field updates. Or use 
let lift?


In [2]:
! verus /tmp/mystep.rs

 --> /tmp/mystep.rs:5:15
  |
5 |     enum PC { mycpy {  },Loop {  },done {  } }
  |               ^^^^^ help: convert the identifier to upper camel case: `Mycpy`
  |
  = note: `#[warn(non_camel_case_types)]` (part of `#[warn(nonstandard_style)]`) on by default

 --> /tmp/mystep.rs:5:36
  |
5 |     enum PC { mycpy {  },Loop {  },done {  } }
  |                                    ^^^^ help: convert the identifier to upper camel case: `Done`

verification results:: 0 verified, 0 errors

 --> /tmp/mystep.rs:5:15
  |
5 |     enum PC { mycpy {  },Loop {  },done {  } }
  |               ^^^^^ help: convert the identifier to upper camel case: `Mycpy`
  |
  = note: `#[warn(non_camel_case_types)]` (part of `#[warn(nonstandard_style)]`) on by default

 --> /tmp/mystep.rs:5:36
  |
5 |     enum PC { mycpy {  },Loop {  },done {  } }
  |                                    ^^^^ help: convert the identifier to upper camel case: `Done`




In [2]:
! cat /tmp/mystep.rs

mod mystep {
    use vstd::prelude::*;
    verus! {

struct State {
    pc: PC,
    ram: Map<u32, u8>,
    len: u32,
    src: u32,
    dst: u32,
}

enum PC {
    mycpy {  },
    Loop {  },
    done {  },
}

spec fn step(st_KDBANG_120: State) -> State {
    (if (st_KDBANG_120.pc == (PC::mycpy {  })) {
        (if (0x0 == st_KDBANG_120.len) {
            (State {
                pc: (PC::done {  }),
                ram: st_KDBANG_120.ram,
                len: st_KDBANG_120.len,
                src: st_KDBANG_120.src,
                dst: st_KDBANG_120.dst,
            })
        } else {
            (State {
                pc: (PC::Loop {  }),
                ram: st_KDBANG_120.ram,
                len: st_KDBANG_120.len,
                src: st_KDBANG_120.src,
                dst: st_KDBANG_120.dst,
            })
        })
    } else {
        (if (st_KDBANG_120.pc == (PC::Loop {  })) {
            (if ((State {
                pc: st_KDBANG_120.pc,
                ram: st_KDBANG_120